# Universidad de la Salle
## Maestría en Inteligencia Artificial
### Fundamentos de Inteligencia Artificial
---
#### Realizado por:
    Camilo Andrés Castañeda Galindo

## ACTIVIDAD 8. Caso de Estudio — Visión por Computador
### Sistema de Control de Calidad de Galletas mediante Visión Artificial
---

**Problema:** Una línea de producción de galletas necesita un sistema automático que identifique en tiempo real si cada galleta está **entera (aprobada)** o **quebrada (rechazada)**, usando únicamente la cámara de un computador.

**Enfoque:** Se utiliza **OpenCV** para capturar video en vivo, aplicar técnicas de procesamiento de imagen y clasificar cada galleta por su **circularidad**:

$$\text{Circularidad} = \frac{4\pi \cdot A}{P^2}$$

Donde $A$ es el área del contorno y $P$ su perímetro. Un círculo perfecto tiene circularidad = 1.0.

| Circularidad | Clasificación |
|:---:|:---:|
| > 0.8 | **Galleta aprobada** (verde) |
| 0.5 – 0.8 | **Galleta quebrada** (rojo) |
| < 0.5 | Descartado (ruido) |

## Pipeline del Sistema

El sistema sigue un flujo de procesamiento clásico en visión por computador:

```
Captura de Video → Escala de Grises → Suavizado Gaussiano → Umbralización (Otsu)
     → Operaciones Morfológicas → Detección de Contornos → Cálculo de Circularidad
          → Clasificación → Visualización en Tiempo Real
```

A continuación se explica cada etapa con su fragmento de código correspondiente.

---
## Paso 1 — Importación de Librerías y Captura de Video

Se utilizan dos librerías:
- **OpenCV (`cv2`):** procesamiento de imágenes y video en tiempo real.
- **NumPy (`np`):** operaciones matemáticas sobre arreglos numéricos.

`cv2.VideoCapture(0)` abre la cámara web predeterminada del equipo.

In [ ]:
import cv2
import numpy as np

# Inicio de captura de video desde la cámara web
capture = cv2.VideoCapture(0)

---
## Paso 2 — Preprocesamiento de la Imagen

Cada fotograma capturado pasa por dos transformaciones:

1. **Escala de grises** (`cvtColor`): reduce la imagen de 3 canales (BGR) a 1 solo canal de intensidad, simplificando el análisis.
2. **Suavizado Gaussiano** (`GaussianBlur`): aplica un filtro de desenfoque con kernel 11×11 para eliminar ruido y pequeñas imperfecciones que podrían afectar la segmentación.

$$G(x, y) = \frac{1}{2\pi\sigma^2} e^{-\frac{x^2 + y^2}{2\sigma^2}}$$

In [ ]:
# Dentro del bucle de captura:
ret, image = capture.read()

# Conversión a escala de grises
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Suavizado Gaussiano (kernel 11x11, sigma=3)
blurred = cv2.GaussianBlur(gray, (11, 11), 3)

---
## Paso 3 — Segmentación por Umbralización (Otsu)

La **umbralización de Otsu** determina automáticamente el valor óptimo que separa el fondo oscuro de la galleta clara. Clasifica cada píxel como **blanco (255)** o **negro (0)**:

$$\text{píxel}(x,y) = \begin{cases} 255 & \text{si } I(x,y) > T_{\text{Otsu}} \\ 0 & \text{en caso contrario} \end{cases}$$

El flag `THRESH_BINARY + THRESH_OTSU` combina el umbralizado binario con el cálculo automático del umbral.

In [ ]:
# Segmentación por umbral automático de Otsu
_, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

---
## Paso 4 — Operaciones Morfológicas

Tras la umbralización, la máscara binaria puede tener bordes dentados y pequeños huecos internos. Se aplican dos operaciones morfológicas con un **elemento estructurante elíptico** de 15×15:

1. **Cierre (`MORPH_CLOSE`)** — 3 iteraciones: rellena hoyos internos y conecta bordes cercanos.
   $$\text{Cierre}(A) = (A \oplus B) \ominus B$$

2. **Apertura (`MORPH_OPEN`)** — 1 iteración: elimina pequeñas protuberancias y ruido residual.
   $$\text{Apertura}(A) = (A \ominus B) \oplus B$$

Donde $\oplus$ es dilatación, $\ominus$ es erosión y $B$ es el kernel elíptico.

In [ ]:
# Elemento estructurante elíptico (15x15)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))

# Cierre morfológico: rellena hoyos y suaviza bordes (3 iteraciones)
thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=3)

# Apertura morfológica: elimina ruido pequeño (1 iteración)
thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

---
## Paso 5 — Detección de Contornos y Clasificación por Circularidad

Se detectan los **contornos externos** de la imagen binaria con `findContours`. Para cada contorno se calcula:

- **Área** ($A$): cantidad de píxeles dentro del contorno.
- **Perímetro** ($P$): longitud del borde del contorno.
- **Circularidad**: qué tan circular es la forma. Un valor de 1.0 indica un círculo perfecto.

**Criterios de clasificación:**
- Se filtran contornos con área > 5000 píxeles (para ignorar ruido).
- Si la circularidad > 0.8 → **Galleta aprobada** (borde verde).
- Si la circularidad está entre 0.5 y 0.8 → **Galleta quebrada** (borde rojo).

In [ ]:
# Encontrar contornos externos
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

for cnt in contours:
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)
    if perimeter == 0:
        continue

    # Cálculo de circularidad
    circularity = 4 * np.pi * area / (perimeter * perimeter)

    # Filtrar: área grande + forma circular
    if area > 5000 and circularity > 0.5:
        (x, y), radius = cv2.minEnclosingCircle(cnt)
        center = (int(x), int(y))
        radius = int(radius)
        cv2.circle(image, center, 1, (255, 0, 0), 3)  # Centro

        if circularity > 0.8:
            label = "Galleta aprobada"
            color = (0, 255, 0)    # Verde
        else:
            label = "Galleta quebrada"
            color = (0, 0, 255)    # Rojo

        # Dibujar círculo envolvente y etiqueta
        cv2.circle(image, center, radius, color, 3)
        cv2.putText(image, f"{label} ({circularity:.2f})",
                    (center[0]-80, center[1]-radius-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

---
## Código Completo

A continuación se presenta el script completo integrado, listo para ejecutarse como archivo `.py` con una cámara web conectada. Presionar la tecla **`q`** cierra la ventana.

In [ ]:
import cv2
import numpy as np

# Inicio de captura de video
capture = cv2.VideoCapture(0)

while True:
    ret, image = capture.read()

    # Preprocesamiento del video
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (11, 11), 3)

    # Segmentación por umbral
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Cerrar el borde dentado y rellenar hoyos de la galleta
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=3)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

    # Encontrar contornos
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)

        # Filtrar: área grande + forma circular
        if area > 5000 and circularity > 0.5:
            (x, y), radius = cv2.minEnclosingCircle(cnt)
            center = (int(x), int(y))
            radius = int(radius)
            cv2.circle(image, center, 1, (255, 0, 0), 3)

            if circularity > 0.8:
                label = "Galleta aprobada"
                color = (0, 255, 0)
            else:
                label = "Galleta quebrada"
                color = (0, 0, 255)

            cv2.circle(image, center, radius, color, 3)
            cv2.putText(image, f"{label} ({circularity:.2f})", (center[0]-80, center[1]-radius-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # Título
    cv2.putText(image, "Control de calidad de galletas", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # Nombre
    h = image.shape[0]
    cv2.putText(image, "Creado por: Camilo A. Castaneda G.", (10, h - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)

    cv2.imshow('video', image)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

---
## Resultados

El sistema fue probado en tiempo real con galletas reales frente a la cámara web. A continuación se muestran los dos escenarios de clasificación:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

%matplotlib inline

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Resultados del Sistema de Control de Calidad', fontsize=15, fontweight='bold')

# Galleta aprobada
img_aprobada = mpimg.imread('galleta_aprobada.png')
axes[0].imshow(img_aprobada)
axes[0].set_title('Galleta Aprobada — Circularidad = 0.89', fontsize=12, color='green', fontweight='bold')
axes[0].axis('off')

# Galleta quebrada
img_quebrada = mpimg.imread('galleta_quebrada.png')
axes[1].imshow(img_quebrada)
axes[1].set_title('Galleta Quebrada — Circularidad = 0.71', fontsize=12, color='red', fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### Análisis de los Resultados

| Escenario | Circularidad | Clasificación | Color del borde |
|:---|:---:|:---:|:---:|
| Galleta entera | **0.89** | Aprobada | Verde |
| Galleta con fragmento roto | **0.71** | Quebrada | Rojo |

- **Galleta aprobada (0.89):** La galleta presenta una forma casi circular, lo que indica que está completa e íntegra. El sistema la marca con un borde **verde** y la etiqueta "Galleta aprobada".
  
- **Galleta quebrada (0.71):** La galleta tiene un pedazo faltante en el borde derecho, lo que reduce notablemente su circularidad. El sistema la detecta correctamente como defectuosa, marcándola con borde **rojo** y la etiqueta "Galleta quebrada".

---
## Conclusiones

### Resumen del Pipeline

| Etapa | Técnica | Propósito |
|:---|:---|:---|
| **Preprocesamiento** | Escala de grises + Gaussiano 11×11 | Reducir ruido y simplificar la imagen |
| **Segmentación** | Umbralización de Otsu | Separar la galleta del fondo automáticamente |
| **Morfología** | Cierre (3 iter.) + Apertura (1 iter.) | Rellenar huecos y eliminar ruido |
| **Detección** | `findContours` + contornos externos | Localizar la silueta de cada galleta |
| **Clasificación** | Circularidad ($4\pi A / P^2$) | Determinar si la galleta está entera o quebrada |

### Hallazgos

1. **La circularidad es un descriptor geométrico sencillo pero efectivo** para distinguir galletas enteras de galletas rotas. Un umbral de 0.8 separa correctamente ambos casos.
2. **Las operaciones morfológicas son fundamentales:** sin el cierre morfológico, los huecos internos (como los puntos decorativos de la galleta) generarían contornos falsos.
3. **El método de Otsu elimina la necesidad de calibrar manualmente el umbral**, lo que hace al sistema robusto ante cambios moderados de iluminación.
4. **El sistema opera en tiempo real**, procesando cada fotograma del video de manera independiente, lo que lo hace adecuado para una línea de producción.

### Ventajas de la Visión por Computador

La inspección visual automatizada mediante técnicas de procesamiento de imágenes permite realizar control de calidad de manera **objetiva, consistente y a alta velocidad**, eliminando la subjetividad y fatiga propias de la inspección humana.